# MultiMedAI — Fine-tune BLIP-VQA on PathVQA (FREE Colab GPU)

**Why:** the local CPU approach (frozen BiomedCLIP + linear head) tops out at ~55% VQA — a documented limit of frozen-feature probes. Reaching **60-80%** needs an **end-to-end** vision-language model with cross-attention between image and question. That needs a GPU, so we do it here on a **free T4** ($0).

**Model:** `Salesforce/blip-vqa-base` (open, BSD) fine-tuned on `flaviagiammarino/path-vqa`. Then download the weights and run inference locally on CPU.

**Honesty:** every accuracy below is real (exact-match on the held-out test split). Reported split into yes/no vs open-ended — yes/no is where 60-80% is realistic; open-ended is genuinely harder (PathVQA SOTA ~50%).

### Run: Colab → Upload → Runtime→T4 GPU → Run all (~40-60 min).

In [ ]:
import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
!pip -q install transformers==4.46.0 datasets==3.0.1 accelerate==1.6.0

## 1. Load BLIP-VQA + PathVQA

In [ ]:
from transformers import BlipProcessor, BlipForQuestionAnswering
from datasets import load_dataset
import torch

proc = BlipProcessor.from_pretrained('Salesforce/blip-vqa-base')
model = BlipForQuestionAnswering.from_pretrained('Salesforce/blip-vqa-base').cuda()

tr = load_dataset('flaviagiammarino/path-vqa', split='train')
te = load_dataset('flaviagiammarino/path-vqa', split='test')
print('train', len(tr), 'test', len(te))

## 2. Baseline (zero-shot, BEFORE fine-tuning) — so improvement is provable

In [ ]:
@torch.no_grad()
def evaluate(model, n=500):
    model.eval(); correct=yn_c=yn_n=oe_c=oe_n=0
    for i in range(n):
        ex = te[i]; a = ex['answer'].strip().lower()
        inp = proc(ex['image'], ex['question'], return_tensors='pt').to('cuda')
        out = model.generate(**inp, max_new_tokens=10)
        pred = proc.decode(out[0], skip_special_tokens=True).strip().lower()
        hit = int(pred == a)
        correct += hit
        if a in ('yes','no'): yn_c+=hit; yn_n+=1
        else: oe_c+=hit; oe_n+=1
    print(f'  overall    {correct/n*100:.2f}%')
    print(f'  yes/no     {yn_c/max(yn_n,1)*100:.2f}%  (n={yn_n})')
    print(f'  open-ended {oe_c/max(oe_n,1)*100:.2f}%  (n={oe_n})')

print('ZERO-SHOT (before fine-tuning):'); evaluate(model)

## 3. Fine-tune on PathVQA

In [ ]:
from torch.utils.data import DataLoader
import torch

N_TRAIN = 19654   # FULL PathVQA train set (max accuracy) — see time note below
EPOCHS  = 5       # more passes; ~1.5-2h on a free T4. Lower to 3 if you're short on time.

def collate(batch):
    imgs = [b['image'] for b in batch]
    qs   = [b['question'] for b in batch]
    ans  = [b['answer'].strip().lower() for b in batch]
    enc = proc(images=imgs, text=qs, return_tensors='pt', padding=True)
    labels = proc.tokenizer(ans, return_tensors='pt', padding=True).input_ids
    enc['labels'] = labels
    return enc

subset = tr.select(range(min(N_TRAIN, len(tr))))
dl = DataLoader(subset, batch_size=16, shuffle=True, collate_fn=collate)
opt = torch.optim.AdamW(model.parameters(), lr=2e-5)
# cosine LR decay over all steps -> more stable convergence for the longer run
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS*len(dl))

model.train()
for epoch in range(EPOCHS):
    tot=0
    for step, batch in enumerate(dl):
        batch = {k:v.cuda() for k,v in batch.items()}
        loss = model(**batch).loss
        loss.backward(); opt.step(); sched.step(); opt.zero_grad()
        tot += loss.item()
        if step % 100 == 0:
            print(f'epoch {epoch+1} step {step} loss {loss.item():.3f}')
    print(f'== epoch {epoch+1} avg loss {tot/len(dl):.3f} ==')
    # save a checkpoint each epoch (free Colab can disconnect on long runs)
    model.save_pretrained('blip_vqa_pathvqa'); proc.save_pretrained('blip_vqa_pathvqa')
    print('  checkpoint saved')

## 4. Accuracy AFTER fine-tuning (the real number)

In [ ]:
print('AFTER fine-tuning:'); evaluate(model, n=1000)

## 5. Save + download the fine-tuned model -> repo `weights/blip_vqa/`

In [ ]:
model.save_pretrained('blip_vqa_pathvqa'); proc.save_pretrained('blip_vqa_pathvqa')
!zip -qr blip_vqa_pathvqa.zip blip_vqa_pathvqa
import os; print('MB:', round(os.path.getsize('blip_vqa_pathvqa.zip')/1e6,1))
try:
    from google.colab import files; files.download('blip_vqa_pathvqa.zip')
except Exception:
    print('Download blip_vqa_pathvqa.zip manually.')

## Done
1. Unzip into your repo: `weights/blip_vqa/` (so `weights/blip_vqa/config.json` etc. exist).
2. The local app's VQA path can load `BlipForQuestionAnswering.from_pretrained('weights/blip_vqa')` and run on CPU (generation is a few seconds).
3. Put the before/after accuracy (overall + yes/no + open-ended) in DEFENSE.md — the zero-shot vs fine-tuned delta is your proof of real training.